# MACE from scratch


The purpose of this notebook is to recreate in a certainly more simple way the MACE framework. The goal of this notebook is to analyze each piece of software with the idea to have a simpler representation of MACE for an educational purpose.

## Geometrical graph construction

### Atomic graph builder

Before MACE can perform message passing, the raw atomic coordinates must be converted into a directed graph. Because molecular dynamics often simulates infinite bulk materials (like liquids or crystals) using a small, repeating simulation box, this graph builder must strictly adhere to Periodic Boundary Conditions (PBC).
- It calculates the perpendicular heights of the simulation box (cell) and dynamically determines how many "ghost" boxes (n_shifts) are needed to cover the cutoff radius in all directions
- It builds a grid of all integer cell shifts, converts them to physical shift vectors via matrix multiplication, and uses PyTorch broadcasting to compute the distance between every atom $i$ and every atom $j$ across all ghost cells simultaneously
- The physical shift vectors are computed using torch.matmul(unit_shifts, cell). This mathematically links the pairwise distances to the cell matrix, allowing PyTorch's autograd to later compute the derivative of the energy with respect to the cell volume
-  It applies a boolean mask to discard interactions beyond the cutoff radius. Crucially, it masks out self-interactions ($i = i$) only within the primary cell (shift [0,0,0]), correctly allowing an atom to interact with its own periodic image if the simulation box is smaller than the cutoff radius
- The module collapses the massive dense distance matrix into sparse graph tensors (edge_index, valid_vectors, valid_distances), ready to be passed into the radial and angular embedding layers

In [ ]:
import torch
import torch.nn as nn

class AtomicGraphBuilder(nn.Module):

    def __init__(self, cutoff: float):
        super().__init__()
        self.cutoff = cutoff

    def forward(self, positions: torch.Tensor, cell: torch.Tensor, pbc: torch.Tensor):

        num_atoms = positions.shape[0]
        device = positions.device

        # Determine how many periodic box replications are needed
        if pbc.any():
            # Calculate the perpendicular widths of the cell
            # Volume = det(cell)
            vol = torch.abs(torch.det(cell))

            # Cross products of cell vectors give the area of the faces
            cross_a = torch.linalg.cross(cell[1], cell[2])
            cross_b = torch.linalg.cross(cell[2], cell[0])
            cross_c = torch.linalg.cross(cell[0], cell[1])

            # Perpendicular heights of the cell
            heights = vol / torch.stack([
                torch.norm(cross_a),
                torch.norm(cross_b),
                torch.norm(cross_c)
            ])

            # Max number of integer shifts needed along each axis
            n_shifts = torch.ceil(self.cutoff / heights).to(torch.int)
            n_shifts = torch.where(pbc, n_shifts, torch.zeros_like(n_shifts))
        else:
            n_shifts = torch.zeros(3, dtype=torch.int, device=device)

        # Generate a grid of all possible integer shifts (S_x, S_y, S_z)
        shift_range_x = torch.arange(-n_shifts[0], n_shifts[0] + 1, device=device)
        shift_range_y = torch.arange(-n_shifts[1], n_shifts[1] + 1, device=device)
        shift_range_z = torch.arange(-n_shifts[2], n_shifts[2] + 1, device=device)

        shift_grid = torch.meshgrid(shift_range_x, shift_range_y, shift_range_z, indexing='ij')
        unit_shifts = torch.stack(shift_grid, dim=-1).view(-1, 3).to(torch.float32)

        # Calculate real-space shift vectors
        shift_vectors = torch.matmul(unit_shifts, cell)

        # Compute all pairwise distances
        pos_i = positions.view(num_atoms, 1, 1, 3)
        pos_j = positions.view(1, num_atoms, 1, 3)

        # Calculate distance vectors
        vectors = pos_j - pos_i + shift_vectors.view(1, 1, -1, 3)
        distances = torch.norm(vectors, dim=-1)

        # Mask out interactions beyond the cutoff and self-interactions
        mask = distances <= self.cutoff

        # Create identity mask for the primary cell (shift=(0,0,0))
        center_shift_idx = (unit_shifts.abs().sum(dim=-1) == 0).nonzero(as_tuple=True)[0]
        eye_mask = torch.eye(num_atoms, dtype=torch.bool, device=device)
        mask[:, :, center_shift_idx] = mask[:, :, center_shift_idx] & (~eye_mask.unsqueeze(-1))

        # Extract the valid edges
        # Find the indices where the mask is True
        idx_i, idx_j, idx_shift = torch.where(mask)

        edge_index = torch.stack([idx_i, idx_j], dim=0)
        valid_vectors = vectors[idx_i, idx_j, idx_shift]
        valid_distances = distances[idx_i, idx_j, idx_shift]
        valid_unit_shifts = unit_shifts[idx_shift]

        return edge_index, valid_vectors, valid_distances, valid_unit_shifts

### Radial envelope

In molecular dynamics (MD), forces are calculated as the negative gradient of the potential energy ($\mathbf{F} = -\nabla E$). If an atom moves across the interaction boundary (e.g., from 4.99 Å to 5.01 Å) and suddenly drops off the neighbor list, the computed energy of the system will experience a discontinuous step-jump. The derivative of a step-jump is infinite, which results in an infinite force spike that will immediately crash the numerical integrators of the simulation.To guarantee physical stability, MACE applies a mathematical stricture called a Radial Envelope to all pairwise interactions before they are processed by the network.

It normalizes the scalar distance $r$ into a dimensionless variable $u = 1 - (r / r_c)$. Thus, $u=1$ when atoms overlap, and $u=0$ at the exact cutoff boundary. Using torch.clamp(u, min=0.0), it acts as a soft mask, safely zeroing out any distances mathematically beyond the cutoff without breaking the backpropagation graph. The normalized distance is raised to a polynomial power. Choosing $p=5$ is deliberate: it guarantees that the energy function ($C^0$), the forces/first derivative ($C^1$), and the Virial stress tensor/second derivative ($C^2$) all decay smoothly and continuously to exactly $0.0$ at the boundary. If $p=1$ were used, the forces would still experience a catastrophic discontinuity.

In [ ]:
import torch
import torch.nn as nn

class RadialEnvelope(nn.Module):

    def __init__(self, cutoff: float, p: int = 5):

        super().__init__()
        self.cutoff = cutoff
        self.p = p

    def forward(self, edge_distances: torch.Tensor) -> torch.Tensor:

        u = 1.0 - (edge_distances / self.cutoff)
        u = torch.clamp(u, min=0.0)
        envelope = u ** self.p

        return envelope

### Bessel radial basis

Standard neural networks struggle to learn complex physical representations from a raw, 1D scalar distance. To provide a rich, continuous feature space for the network to optimize, the scalar distance must be expanded into a high-dimensional vector.This module projects the pairwise distance onto a set of Spherical Bessel functions and then mixes them using a Multi-Layer Perceptron (MLP) to create learnable radial features.
The MLP used to mix the raw Bessel functions is instantiated with bias=False on all linear layers. If a bias term were used, the network would constantly add a baseline offset to the features. When an atom crosses the cutoff boundary, the distance features would not evaluate to zero, completely destroying  constraints established by the envelope. The module processes the features through the neural network first, and only multiplies the output by the RadialEnvelope at the very end. Regardless of the internal weights the MLP learns during training, all features are so rigorously forced to $0.0$ at the cutoff boundary.

In [ ]:
import torch
import torch.nn as nn
import math

class BesselRadialBasis(nn.Module):

    def __init__(self, num_basis: int, cutoff: float, radial_channels: int, envelope_module: nn.Module):

        super().__init__()
        self.num_basis = num_basis
        self.cutoff = cutoff
        self.envelope = envelope_module

        # Initialize the frequency multipliers for the Bessel functions
        frequencies = torch.arange(1, num_basis + 1, dtype=torch.float32)
        self.register_buffer("frequencies", frequencies)

        self.mlp = nn.Sequential(
            nn.Linear(num_basis, 64, bias=False),
            nn.SiLU(),
            nn.Linear(64, radial_channels, bias=False)
        )

    def forward(self, edge_distances: torch.Tensor) -> torch.Tensor:

        # Ensure distances has an explicit feature dimension
        r = edge_distances.unsqueeze(-1)

        #  Compute raw Bessel functions

        x = self.frequencies * r / self.cutoff

        # Prefactor derived from rearranging the formula to use sinc
        prefactor = (math.pi * self.frequencies) / self.cutoff * math.sqrt(2.0 / self.cutoff)

        # Compute safe Bessel functions
        bessel = prefactor * torch.sinc(x)

        #  Make the features learnable
        learned_features = self.mlp(bessel)

        # Apply the physical envelope stricture
        # embedding is mathematically forced to 0.0 at the cutoff distance
        env = self.envelope(edge_distances).unsqueeze(-1) # shape (num_edges, 1)

        radial_embedding = learned_features * env

        return radial_embedding

### Sperical harmonics basis

If a standard neural network is fed raw coordinate vectors, it will predict completely different energies if the molecular system is simply rotated in 3D space. To fix this, traditional machine learning relies on massive data augmentation (showing the network millions of rotated molecules).MACE completely bypasses this by translating the 3D geometry into Spherical Harmonics ($Y_l^m$), the exact mathematical functions that describe the shapes of quantum electron orbitals .By using Spherical Harmonics, the network achieves strict $O(3)$ Equivariance. If the input molecule is rotated, the internal mathematical features of the network rotate perfectly and predictably via Wigner-D matrices, meaning the network mathematically "understands" 3D space without needing to learn it from data. he module uses the e3nn group theory library to generate features based on a maximum angular momentum ($L_{max}$). The result is the Angular Embedding, a tensor that will be combined with the radial embeddings and chemical nodes to build the complete many body physics messages.

In [ ]:
import torch
import torch.nn as nn
from e3nn import o3

class SphericalHarmonicsBasis(nn.Module):

    def __init__(self, lmax: int):

        super().__init__()
        self.lmax = lmax
        self.sh_irreps = o3.Irreps.spherical_harmonics(lmax)

    def forward(self, edge_vectors: torch.Tensor) -> torch.Tensor:
        # Normalize the vectors
        # Spherical harmonics only care about direction
        edge_lengths = torch.norm(edge_vectors, dim=1, keepdim=True)
        normalized_vectors = edge_vectors / (edge_lengths + 1e-6)

        # Compute the Spherical Harmonics
        sh_embedding = o3.spherical_harmonics(
            self.sh_irreps,
            normalized_vectors,
            normalize=False # We already normalized manually for safety
        )

        return sh_embedding

### Chemical Embedding

This class acts as the translation layer that converts raw atomic chemistry (integer atomic numbers) into the continuous mathematical format required by an $E(3)$-equivariant neural network. The nn.Embedding acts as a lookup table, assigning a unique, learnable list of continuous numbers to each element. This allows the network to learn that Tin and Iodine are chemically different, but perhaps mathematically related in the latent space. Because MACE uses the e3nn library, all hidden features must obey 3D rotational symmetries. Features are divided by angular momentum (L).

In [ ]:
import torch
import torch.nn as nn
from e3nn import o3

class ChemicalEmbedding(nn.Module):

    def __init__(self, num_elements: int, node_irreps: str | o3.Irreps):

        super().__init__()
        self.node_irreps = o3.Irreps(node_irreps)
        self.scalar_dim = sum(mul for mul, ir in self.node_irreps if ir.l == 0)
        self.total_dim = self.node_irreps.dim

        #  The embedding layer only learns features for the scalar channels
        self.embedding = nn.Embedding(num_elements, self.scalar_dim)

    def forward(self, atomic_numbers: torch.Tensor) -> torch.Tensor:
        num_atoms = atomic_numbers.shape[0]

        # Look up the scalar features for each atom
        scalar_features = self.embedding(atomic_numbers)

        # Create a full tensor of zeros for all channels
        node_features = torch.zeros(
            num_atoms,
            self.total_dim,
            device=atomic_numbers.device,
            dtype=scalar_features.dtype
        )

        # Inject the scalar features into the corresponding indices.
        node_features[:, :self.scalar_dim] = scalar_features

        return node_features

### One Particle Basis

Instead of looking at complex combinations of atoms right away, this class first collapses all neighboring atoms into a single, continuous mathematical cloud (density) around the central atom.
- The network calculates how far away the neighbor is ($r_{ij}$) and passes that distance through a linear layer. This generates a unique set of dynamic weights that tell the tensor product how strongly this specific neighbor should influence the central atom based purely on its distance.
- It takes the current chemical identity/state of the neighbor, the spherical harmonics describing the exact 3D angle/direction to the neighbor ($Y_{lm}$), the spherical harmonics describing the exact 3D angle/direction to the neighbor ($Y_{lm}$) and pools them into the A-basis.

Instead of keeping all these neighbor messages separate (which would cause the exponential scaling problem later), the index_add_ function sums them all together onto the central atom $i$.By adding all the 2-body messages together, the network merges the individual neighbors into a single, continuous neighborhood density (the A-basis). This output tensor is exactly what MACE will take in the next step and multiply by itself (squaring or cubing) to efficiently generate the 3-body and 4-body angular correlations without ever having to loop over pairs or triplets of atoms.

In [ ]:
import torch
import torch.nn as nn
from e3nn import o3

class OneParticleBasis(nn.Module):
    def __init__(self, node_irreps: o3.Irreps, sh_irreps: o3.Irreps, radial_channels: int):

        super().__init__()
        self.node_irreps = o3.Irreps(node_irreps)
        self.sh_irreps = o3.Irreps(sh_irreps)

        #  The Clebsch-Gordan Tensor Product
        self.tp = o3.FullyConnectedTensorProduct(
            irreps_in1=self.node_irreps,
            irreps_in2=self.sh_irreps,
            irreps_out=self.node_irreps,
            shared_weights=False, # Weights will be provided dynamically per-edge
            internal_weights=False
        )

        # Radial Weight Generator
        self.weight_net = nn.Linear(radial_channels, self.tp.weight_numel, bias=False)

    def forward(self, node_features: torch.Tensor,
                sh_embedding: torch.Tensor,
                radial_embedding: torch.Tensor,
                edge_index: torch.Tensor) -> torch.Tensor:

        idx_i, idx_j = edge_index[0], edge_index[1]

        #  Generate the weights for this specific distance
        tp_weights = self.weight_net(radial_embedding)

        messages = self.tp(
            node_features[idx_j], # Neighbor's features
            sh_embedding,         # Angular direction to neighbor
            tp_weights            # Distance weighting
        )

        # Pool the messages onto the receiving atom
        num_atoms = node_features.shape[0]
        A_basis = torch.zeros(
            num_atoms,
            self.node_irreps.dim,
            device=node_features.device,
            dtype=node_features.dtype
        )

        # Sum all incoming edge messages onto node i
        A_basis.index_add_(0, idx_i, messages)

        return A_basis

### Symmetric contraction

This module is the heart of the MACE architecture. It represents the "ACE" (Atomic Cluster Expansion) step.In older graph neural networks, if you wanted to understand how three atoms interact (a 3-body effect, like bond-angle bending), you had to add more message-passing layers so that Atom A could send a message to Atom B, and then Atom B could pass it to Atom C. To understand 4-body physics, you needed even more layers. This is computationally slow and causes the network to "blur" features over long distances. Because the OneParticleBasis ($A$) already contains the full geometric "halo" of all neighbors, MACE can compute $N$-body interactions strictly on the central atom without looking across the graph again. It does this by mathematically multiplying the $A$-basis by itself:$\nu = 1$ (2-body): The $A$-basis itself.$\nu = 2$ (3-body): $A \otimes A$ (multiplying the local environment by itself).$\nu = 3$ (4-body): $A \otimes A \otimes A$.By iteratively multiplying these equivariant tensors using Clebsch-Gordan coefficients, the network builds complex, many-body correlations while maintaining perfect $O(3)$ rotational symmetry.

In [ ]:
import torch
import torch.nn as nn
from e3nn import o3

class SymmetricContraction(nn.Module):
    def __init__(self, node_irreps: o3.Irreps, correlation_order: int):

        super().__init__()
        self.correlation_order = correlation_order
        self.node_irreps = o3.Irreps(node_irreps)

        #  Iterative Tensor Products
        # We need a sequence of tensor products to build higher-order correlations.
        self.tensor_products = nn.ModuleList()
        for _ in range(correlation_order - 1):
            # We project the output back down to the original node_irreps
            # to prevent the number of channels from exploding exponentially
            tp = o3.FullyConnectedTensorProduct(
                irreps_in1=self.node_irreps,
                irreps_in2=self.node_irreps,
                irreps_out=self.node_irreps
            )
            self.tensor_products.append(tp)

        # Linear Mixing Layer
        concat_irreps = self.node_irreps * correlation_order
        self.linear_mix = o3.Linear(
            irreps_in=concat_irreps,
            irreps_out=self.node_irreps
        )

    def forward(self, A_basis: torch.Tensor) -> torch.Tensor:
        # Store each order of correlation starting with the 2-body term (v=1)
        # B_orders = [A]
        B_orders = [A_basis]

        current_B = A_basis

        # Iteratively multiply the highest computed order by the A-basis again
        for tp in self.tensor_products:
            # v=2: current_B = A (x) A
            # v=3: current_B = A^2 (x) A
            current_B = tp(current_B, A_basis)
            B_orders.append(current_B)

        # Concatenate all orders together along the feature dimension
        # Shape: (num_atoms, feature_dim * correlation_order)
        B_concat = torch.cat(B_orders, dim=-1)

        # Linearly mix the concatenated many-body features back into the standard size
        B_basis = self.linear_mix(B_concat)

        return B_basis

### Equivariant Update

Neural network with common activations predicts different physics depending on how you rotate the molecule in space. This completely destroys the $O(3)$ equivariance. You cannot alter $X, Y$, and $Z$ independently. To introduce non-linearity safely, MACE uses a Gate. The code looks at the atom's internal state and strictly separates the non-directional features from the directional ones, with the goal to apply a standard activation to scalar features and a separated approach to directional features.


In [ ]:
import torch
import torch.nn as nn
from e3nn import o3
from e3nn.nn import Gate

class EquivariantUpdate(nn.Module):
    def __init__(self, node_irreps: o3.Irreps):

        super().__init__()
        self.node_irreps = o3.Irreps(node_irreps)

        #  Self-Interaction Layer
        # Processes the atom's previous state before adding the new messages
        self.self_interaction = o3.Linear(self.node_irreps, self.node_irreps)

        # 2. Setup the Gated Non-Linearity
        # We must split the irreps into scalars (L=0) and non-scalars (L>0).
        irreps_scalars = o3.Irreps([ir for ir in self.node_irreps if ir.ir.l == 0])
        irreps_gated = o3.Irreps([ir for ir in self.node_irreps if ir.ir.l > 0])

        # We need to generate one scalar gate for every non-scalar channel we want to scale.
        irreps_gates = o3.Irreps(f"{irreps_gated.num_irreps}x0e")

        # Linear projection to shape the B-basis into the exact dimensions the Gate expects
        self.linear_before_gate = o3.Linear(
            self.node_irreps,
            irreps_scalars + irreps_gates + irreps_gated
        )

        # The e3nn Gate safely applies non-linearities without breaking physics
        self.gate = Gate(
            irreps_scalars=irreps_scalars,
            act_scalars=[nn.SiLU()],      # Standard activation for scalars
            irreps_gates=irreps_gates,
            act_gates=[nn.Sigmoid()],     # 0 to 1 scaling gates for vectors/tensors
            irreps_gated=irreps_gated     # The vectors/tensors that will be multiplied
        )

        # Final linear projection back to the standard node_irreps dimension
        self.linear_after_gate = o3.Linear(self.gate.irreps_out, self.node_irreps)

    def forward(self, old_node_features: torch.Tensor, B_basis: torch.Tensor) -> torch.Tensor:
        #  Process the atom's previous internal state
        residual = self.self_interaction(old_node_features)

        #  Apply Equivariant Non-Linearities to the new many-body messages
        # Expand the B-basis to create the scalar channels, the gates, and the vectors
        x = self.linear_before_gate(B_basis)

        # Apply the SiLU to scalars, and scale the vectors by the Sigmoid gates
        x = self.gate(x)

        # Compress back to standard dimensionality
        delta_h = self.linear_after_gate(x)

        # Residual Connection
        # Add the new, non-linear many-body features to the old state
        new_node_features = residual + delta_h

        return new_node_features

### Atomic readout

At the end of a MACE layer, the network has successfully updated every atom's hidden state with complex, many-body, rotationally equivariant geometry.

In [ ]:
import torch
import torch.nn as nn
from e3nn import o3

class AtomicReadout(nn.Module):

    def __init__(self, node_irreps: o3.Irreps, hidden_dim: int = 16):
        super().__init__()
        self.node_irreps = o3.Irreps(node_irreps)

        #  Identify the Scalar Channels
        self.scalar_irreps = o3.Irreps([ir for ir in self.node_irreps if ir.ir.l == 0])

        #  Equivariant Filter
        self.equiv_projector = o3.Linear(self.node_irreps, self.scalar_irreps)

        #  Energy MLP
        self.energy_mlp = nn.Sequential(
            nn.Linear(self.scalar_irreps.dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1) # Output a single scalar number (Energy)
        )

    def forward(self, node_features: torch.Tensor) -> torch.Tensor:

        #  Filter out the vectors and tensors.
        scalars = self.equiv_projector(node_features)

        #  Predict the local energy for each atom.
        atomic_energies = self.energy_mlp(scalars)

        return atomic_energies

### MACE force wrapper

In [ ]:
import torch
import torch.nn as nn
from e3nn import o3

class MACELayer(nn.Module):

    def __init__(self, node_irreps, sh_irreps, radial_channels, correlation_order):
        super().__init__()
        self.one_particle = OneParticleBasis(node_irreps, sh_irreps, radial_channels)
        self.symmetric_contraction = SymmetricContraction(node_irreps, correlation_order)
        self.equiv_update = EquivariantUpdate(node_irreps)
        self.readout = AtomicReadout(node_irreps)

    def forward(self, node_features, sh, rad, edge_index):
        # 1. 2-Body Density Projection
        A = self.one_particle(node_features, sh, rad, edge_index)
        # 2. Many-Body Expansion
        B = self.symmetric_contraction(A)
        # 3. Equivariant State Update
        updated_nodes = self.equiv_update(node_features, B)
        # 4. Energy Readout
        energies = self.readout(updated_nodes)

        return updated_nodes, energies


class MACEForceWrapper(nn.Module):
    """
    Assembles the graph, routes data through the embeddings
    and layers, and uses autograd to compute exact, energy-conserving forces.
    """
    def __init__(self, num_elements=100, cutoff=5.0, lmax=2, correlation_order=3, num_layers=2):
        super().__init__()

        # 1. Network Hyperparameters
        self.node_irreps = o3.Irreps("16x0e + 16x1o + 16x2e")
        self.sh_irreps = o3.Irreps.spherical_harmonics(lmax)
        self.radial_channels = 32

        # 2. Instantiating the Geometric Pipeline (Modules 1-5)
        self.graph_builder = AtomicGraphBuilder(cutoff=cutoff)
        self.radial_envelope = RadialEnvelope(cutoff=cutoff, p=5)
        self.radial_basis = BesselRadialBasis(num_basis=8, cutoff=cutoff,
                                              radial_channels=self.radial_channels,
                                              envelope_module=self.radial_envelope)
        self.sh_basis = SphericalHarmonicsBasis(lmax=lmax)
        self.chem_embedding = ChemicalEmbedding(num_elements=num_elements, node_irreps=self.node_irreps)

        # 3. Instantiating the Message Passing Layers (Modules 6-9)
        self.layers = nn.ModuleList([
            MACELayer(self.node_irreps, self.sh_irreps, self.radial_channels, correlation_order)
            for _ in range(num_layers)
        ])

    def forward(self, positions: torch.Tensor, cell: torch.Tensor, pbc: torch.Tensor, atomic_numbers: torch.Tensor):

        positions.requires_grad_(True)

        # 1. Build the Topology
        edge_index, edge_vectors, edge_lengths, _ = self.graph_builder(positions, cell, pbc)

        # 2. Build the Embeddings
        sh = self.sh_basis(edge_vectors)          # Angular (Direction)
        rad = self.radial_basis(edge_lengths)     # Radial (Distance)
        node_features = self.chem_embedding(atomic_numbers) # Chemical (Identity)

        # 3. Message Passing and Energy Accumulation
        total_energy = 0.0

        for layer in self.layers:
            # Update the atom states AND extract the local energy contribution
            node_features, layer_energies = layer(node_features, sh, rad, edge_index)

            # Sum the local atomic energies into the system's Total Energy
            total_energy = total_energy + layer_energies.sum()


        forces = -torch.autograd.grad(
            outputs=total_energy,
            inputs=positions,
            grad_outputs=torch.ones_like(total_energy),
            create_graph=True,
            retain_graph=True
        )[0]

        return total_energy, forces

RMS Forces scaling

In [ ]:
import os
import torch
import numpy as np
import ase.io
from ase.io import read
from google.colab import drive
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define all file paths pointing to the MDProject folder
base_dir = '/content/drive/MyDrive/MDProject'

outcar_path = os.path.join(base_dir, 'OUTCAR')
cleaned_path = os.path.join(base_dir, 'cleaned_dataset.extxyz')
train_path = os.path.join(base_dir, 'train.extxyz')
valid_path = os.path.join(base_dir, 'valid.extxyz')

try:
    print(f"Reading from: {outcar_path} ...")
    atoms_list = ase.io.read(outcar_path, index=':')
    print(f"Retrieved {len(atoms_list)} structures from OUTCAR.")

    # Save the full usable dataset
    ase.io.write(cleaned_path, atoms_list)
    print(f"Full dataset saved as {cleaned_path}")

    # Shuffle and Split (using the atoms_list already in memory)
    np.random.shuffle(atoms_list)

    split = int(0.9 * len(atoms_list))
    train_frames = atoms_list[:split]
    valid_frames = atoms_list[split:]

    # Save the train and validation sets
    ase.io.write(train_path, train_frames)
    ase.io.write(valid_path, valid_frames)

    print(f"Saved {len(train_frames)} frames in {train_path}")
    print(f"Saved {len(valid_frames)} frames in {valid_path}")

except Exception as e:
    print(f"Error processing files: {e}")


def load_extxyz_to_pyg(file_path: str):
    """Parses an .extxyz file and converts it into a list of PyTorch Geometric graphs."""
    structures = read(file_path, index=":")
    dataset = []

    for atoms in structures:
        # Extract features
        positions = torch.tensor(atoms.get_positions(), dtype=torch.float32)
        atomic_numbers = torch.tensor(atoms.get_atomic_numbers(), dtype=torch.long)

        # Extract periodic boundary conditions and simulation cell
        cell = torch.tensor(atoms.get_cell().array, dtype=torch.float32)
        pbc = torch.tensor(atoms.get_pbc(), dtype=torch.bool)

        # Extract ground-truth targets safely handling ASE warnings
        energy_val = atoms.info.get('REF_energy', atoms.info.get('energy', 0.0))
        energy = torch.tensor([energy_val], dtype=torch.float32)

        if 'REF_forces' in atoms.arrays:
            forces_val = atoms.arrays['REF_forces']
        elif 'forces' in atoms.arrays:
            forces_val = atoms.arrays['forces']
        else:
            forces_val = atoms.get_forces()
        forces = torch.tensor(forces_val, dtype=torch.float32)

        # Package into a PyG Data object
        data = Data(
            positions=positions,
            atomic_numbers=atomic_numbers,
            cell=cell,
            pbc=pbc,
            REF_energy=energy,
            REF_forces=forces
        )
        dataset.append(data)

    return dataset

def calculate_rms_forces(train_extxyz_path: str) -> float:
    """Calculates the Root Mean Square of all atomic forces in the training set."""
    train_structures = read(train_extxyz_path, index=":")
    all_forces = []

    for atoms in train_structures:
        if 'REF_forces' in atoms.arrays:
            forces = atoms.arrays['REF_forces']
        elif 'forces' in atoms.arrays:
            forces = atoms.arrays['forces']
        else:
            forces = atoms.get_forces()

        all_forces.append(forces)

    all_forces_np = np.concatenate(all_forces, axis=0)
    rms = np.sqrt(np.mean(all_forces_np ** 2))
    return float(rms)


print("\nLoading datasets into PyTorch Geometric format...")
train_dataset = load_extxyz_to_pyg(train_path)
valid_dataset = load_extxyz_to_pyg(valid_path)

# Batch the data for the GPU
train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=5, shuffle=False)
print(f"Loaded {len(train_dataset)} training structures into {len(train_loader)} batches.")

# Execute the precalculation using the Google Drive path!
rms_forces_scalar = calculate_rms_forces(train_path)
print(f"Precalculated RMS Forces: {rms_forces_scalar:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading from: /content/drive/MyDrive/MDProject/OUTCAR ...
Retrieved 10000 structures from OUTCAR.
Full dataset saved as /content/drive/MyDrive/MDProject/cleaned_dataset.extxyz
Saved 9000 frames in /content/drive/MyDrive/MDProject/train.extxyz
Saved 1000 frames in /content/drive/MyDrive/MDProject/valid.extxyz

Loading datasets into PyTorch Geometric format...
Loaded 9000 training structures into 1800 batches.
Precalculated RMS Forces: 0.3001


Loss function

In [ ]:
import torch.nn.functional as F

def universal_loss(pred_energy, true_energy, pred_forces, true_forces, rms_scaling_factor):

    # Square the RMS factor to get the variance denominator
    scale = rms_scaling_factor ** 2

    #  Compute standard MSE and divide by the scale
    # The weights 1.0 (energy) and 100.0 (forces) match your custom_args
    loss_e = (F.mse_loss(pred_energy, true_energy) / scale) * 1.0
    loss_f = (F.mse_loss(pred_forces, true_forces) / scale) * 100.0

    total_loss = loss_e + loss_f

    return total_loss

Optimizer

Import input files

Training of the simplified MACE structure

In [ ]:
import os
import numpy as np
import ase.io
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define all file paths pointing to the MDProject folder
base_dir = '/content/drive/MyDrive/MDProject'

outcar_path = os.path.join(base_dir, 'OUTCAR')
cleaned_path = os.path.join(base_dir, 'cleaned_dataset.extxyz')
train_path = os.path.join(base_dir, 'train.extxyz')
valid_path = os.path.join(base_dir, 'valid.extxyz')

try:
    # 3. Read OUTCAR
    print(f"Reading from: {outcar_path} ...")
    atoms_list = ase.io.read(outcar_path, index=':')
    print(f"Retrieved {len(atoms_list)} structures from OUTCAR.")

    # 4. Save the full usable dataset
    ase.io.write(cleaned_path, atoms_list)
    print(f"Full dataset saved as {cleaned_path}")

    # 5. Shuffle and Split (using the atoms_list already in memory)
    np.random.shuffle(atoms_list)

    split = int(0.9 * len(atoms_list))
    train_frames = atoms_list[:split]
    valid_frames = atoms_list[split:]

    # 6. Save the train and validation sets
    ase.io.write(train_path, train_frames)
    ase.io.write(valid_path, valid_frames)

    print(f"Saved {len(train_frames)} frames in {train_path}")
    print(f"Saved {len(valid_frames)} frames in {valid_path}")

except Exception as e:
    print(f"Error processing files: {e}")

train_file = train_path
valid_file = valid_path

# 2. Load the data into PyTorch Geometric
train_dataset = load_extxyz_to_pyg(train_file)
valid_dataset = load_extxyz_to_pyg(valid_file)


# Batch size of 5 as specified in the logs
train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=5, shuffle=False)

# 3. Precalculate RMS Forces for Scaling
rms_forces_scalar = calculate_rms_forces(train_file) # Function defined in previous step
print(f"RMS Forces Scalar: {rms_forces_scalar:.4f}")

# 4. Initialize the custom MACE Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MACEForceWrapper(
    num_elements=100,
    cutoff=5.0,
    lmax=0, # max_L=0 from notebook custom_args
    correlation_order=3,
    num_layers=2
).to(device)

# 5. Shuffle and Split (using the atoms_list already in memory)
np.random.shuffle(atoms_list)

# --- DEFINE YOUR REDUCED DATASET SIZES HERE ---
NUM_TRAIN = 500  # Number of structures to train on
NUM_VALID = 50   # Number of structures to validate on

 # Slice the exact number of frames you want
train_frames = atoms_list[:NUM_TRAIN]
valid_frames = atoms_list[NUM_TRAIN : NUM_TRAIN + NUM_VALID]

# 6. Save the train and validation sets
ase.io.write(train_path, train_frames)
ase.io.write(valid_path, valid_frames)

print(f"Saved {len(train_frames)} frames in {train_path}")
print(f"Saved {len(valid_frames)} frames in {valid_path}")
# 5. Setup Optimizer and EMA
optimizer = AdamW(model.parameters(), lr=0.0001, weight_decay=5e-7, amsgrad=True)
ema = ExponentialMovingAverage(model.parameters(), decay=0.99)
max_grad_norm = 10.0
epochs = 50

for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        batch_pred_energies = []
        batch_pred_forces = []

        for data in batch.to_data_list():
            # Now data.cell is safely guaranteed to be (3, 3)
            pred_energy, pred_forces = model(
                positions=data.positions,
                cell=data.cell.view(3, 3), # Explicitly ensure 3x3 shape
                pbc=data.pbc,
                atomic_numbers=data.atomic_numbers
            )
            batch_pred_energies.append(pred_energy)
            batch_pred_forces.append(pred_forces)

        # 1. Stack the 0D energy scalars into a 1D tensor, then force
        # the shape to match the batch.REF_energy target exactly.
        pred_energy = torch.stack(batch_pred_energies).view_as(batch.REF_energy)

        # 2. Concatenate the 2D force arrays normally. This accurately mimics
        # how PyTorch Geometric stacks atom-level features.
        pred_forces = torch.cat(batch_pred_forces)

        # Calculate Universal Loss
        loss = universal_loss(
            pred_energy=pred_energy,
            true_energy=batch.REF_energy,
            pred_forces=pred_forces,
            true_forces=batch.REF_forces,
            rms_scaling_factor=rms_forces_scalar
        )

        # Backpropagation
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()

        # Update Exponential Moving Average
        ema.update()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f}")

# 7. Apply EMA weights for inference/saving
ema.copy_to(model.parameters())
torch.save(model.state_dict(), "./new_simulation/1.NVT_300/A.1_10K/checkpoints/custom_mace_final.pt")
print("Training Complete. Model saved.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading from: /content/drive/MyDrive/MDProject/OUTCAR ...
Retrieved 10000 structures from OUTCAR.
Full dataset saved as /content/drive/MyDrive/MDProject/cleaned_dataset.extxyz
Saved 9000 frames in /content/drive/MyDrive/MDProject/train.extxyz
Saved 1000 frames in /content/drive/MyDrive/MDProject/valid.extxyz
RMS Forces Scalar: 0.3002
Saved 500 frames in /content/drive/MyDrive/MDProject/train.extxyz
Saved 50 frames in /content/drive/MyDrive/MDProject/valid.extxyz
Epoch 1/50 | Train Loss: 1729.7217


KeyboardInterrupt: 

In [ ]:
import os
import torch
import numpy as np
import ase.io
from ase.io import read
from google.colab import drive
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch.optim import AdamW
from torch_ema import ExponentialMovingAverage

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define all file paths pointing to the MDProject folder
base_dir = '/content/drive/MyDrive/MDProject'

outcar_path = os.path.join(base_dir, 'OUTCAR')
cleaned_path = os.path.join(base_dir, 'cleaned_dataset.extxyz')
train_path = os.path.join(base_dir, 'train.extxyz')
valid_path = os.path.join(base_dir, 'valid.extxyz')

try:
    print(f"Reading from: {outcar_path} ...")
    atoms_list = ase.io.read(outcar_path, index=':')
    print(f"Retrieved {len(atoms_list)} structures from OUTCAR.")

    # 1. Extract energies to sort the dataset
    energies = []
    for atoms in atoms_list:
        e = atoms.info.get('REF_energy', atoms.info.get('energy', 0.0))
        energies.append(e)

    # 2. Sort atoms from lowest energy (most stable) to highest energy
    sorted_atoms = [atoms for _, atoms in sorted(zip(energies, atoms_list), key=lambda pair: pair[0])]

    # 3. EXCLUDE HIGH-ENERGY FRAMES: Drop the top 5% most energetic outliers
    keep_cutoff = int(0.95 * len(sorted_atoms))
    filtered_atoms = sorted_atoms[:keep_cutoff]
    print(f"Excluded {len(sorted_atoms) - keep_cutoff} high-energy outlier frames.")

    # 4. Shuffle the clean dataset for proper I.I.D. batching
    np.random.shuffle(filtered_atoms)

    # --- APPLY REDUCED DATASET SIZES ---
    NUM_TRAIN = 500  # Number of structures to train on
    NUM_VALID = 50   # Number of structures to validate on

    # Slice the exact number of frames you want from the clean, shuffled list
    train_frames = filtered_atoms[:NUM_TRAIN]
    valid_frames = filtered_atoms[NUM_TRAIN : NUM_TRAIN + NUM_VALID]

    # Save the strictly reduced train and validation sets
    ase.io.write(train_path, train_frames)
    ase.io.write(valid_path, valid_frames)

    print(f"Saved {len(train_frames)} frames in {train_path}")
    print(f"Saved {len(valid_frames)} frames in {valid_path}")

except Exception as e:
    print(f"Error processing files: {e}")

print("\nLoading datasets into PyTorch Geometric format...")
train_dataset = load_extxyz_to_pyg(train_path)
valid_dataset = load_extxyz_to_pyg(valid_path)

# Batch size of 5
train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=5, shuffle=False)
print(f"Loaded {len(train_dataset)} training structures into {len(train_loader)} batches.")

# Precalculate RMS Forces for Scaling
rms_forces_scalar = calculate_rms_forces(train_path)
print(f"RMS Forces Scalar: {rms_forces_scalar:.4f}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MACEForceWrapper(
    num_elements=100,
    cutoff=5.0,
    lmax=0,
    correlation_order=3,
    num_layers=2
).to(device)

optimizer = AdamW(model.parameters(), lr=0.0001, weight_decay=5e-7, amsgrad=True)
ema = ExponentialMovingAverage(model.parameters(), decay=0.99)
max_grad_norm = 10.0
epochs = 50

print("\nStarting Training Loop...")
for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        batch_pred_energies = []
        batch_pred_forces = []

        # Unbatch the super-graph to prevent cross-molecule bonding
        for data in batch.to_data_list():
            pred_energy, pred_forces = model(
                positions=data.positions,
                cell=data.cell.view(3, 3),
                pbc=data.pbc,
                atomic_numbers=data.atomic_numbers
            )
            batch_pred_energies.append(pred_energy)
            batch_pred_forces.append(pred_forces)

        # Reconstruct batched tensors for loss calculation
        pred_energy = torch.stack(batch_pred_energies).view_as(batch.REF_energy)
        pred_forces = torch.cat(batch_pred_forces)

        # Calculate Universal Loss
        loss = universal_loss(
            pred_energy=pred_energy,
            true_energy=batch.REF_energy,
            pred_forces=pred_forces,
            true_forces=batch.REF_forces,
            rms_scaling_factor=rms_forces_scalar
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()
        ema.update()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f}")

# Apply EMA weights for final model
ema.copy_to(model.parameters())
os.makedirs(f"{base_dir}/checkpoints", exist_ok=True)
torch.save(model.state_dict(), f"{base_dir}/checkpoints/custom_mace_final.pt")
print("Training Complete. Model saved.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading from: /content/drive/MyDrive/MDProject/OUTCAR ...
Retrieved 10000 structures from OUTCAR.
Saved 500 frames in /content/drive/MyDrive/MDProject/train.extxyz
Saved 50 frames in /content/drive/MyDrive/MDProject/valid.extxyz

Loading datasets into PyTorch Geometric format...
Loaded 500 training structures into 100 batches.
RMS Forces Scalar: 0.3015

Starting Training Loop...
Epoch 1/50 | Train Loss: 271.2080
Epoch 2/50 | Train Loss: 100.0091
Epoch 3/50 | Train Loss: 99.9344
Epoch 4/50 | Train Loss: 99.8234
Epoch 5/50 | Train Loss: 99.6660
Epoch 6/50 | Train Loss: 99.2244
Epoch 7/50 | Train Loss: 97.8701
Epoch 8/50 | Train Loss: 96.0806
Epoch 9/50 | Train Loss: 93.7871
Epoch 10/50 | Train Loss: 91.2822
Epoch 11/50 | Train Loss: 88.6848
Epoch 12/50 | Train Loss: 86.3256
Epoch 13/50 | Train Loss: 84.9373
Epoch 14/50 | Train Loss: 83.3565
Epoch 15/50 | Train 

In [ ]:
from ase.calculators.calculator import Calculator, all_changes
import ase.units as units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.io.trajectory import Trajectory

class CustomMACECalculator(Calculator):
    """Bridges the PyTorch MACE model with the ASE Molecular Dynamics engine."""
    implemented_properties = ['energy', 'forces']

    def __init__(self, model, device, **kwargs):
        super().__init__(**kwargs)
        self.model = model
        self.device = device

        # Switch model to evaluation mode (turns off dropout, locks weights)
        self.model.eval()

    def calculate(self, atoms=None, properties=['energy', 'forces'], system_changes=all_changes):
        super().calculate(atoms, properties, system_changes)

        # 1. Extract standard numpy arrays from ASE
        pos = torch.tensor(atoms.get_positions(), dtype=torch.float32, device=self.device)
        numbers = torch.tensor(atoms.get_atomic_numbers(), dtype=torch.long, device=self.device)
        cell = torch.tensor(atoms.get_cell().array, dtype=torch.float32, device=self.device)
        pbc = torch.tensor(atoms.get_pbc(), dtype=torch.bool, device=self.device)

        # 2. Calculate Energy and Forces
        # Crucial: We must keep grad_enabled(True) because our MACE model
        # calculates forces by taking the autograd derivative of the energy!
        with torch.set_grad_enabled(True):
            energy_tensor, forces_tensor = self.model(pos, cell, pbc, numbers)

            # 3. Pass the results back to ASE in standard numpy format
            self.results['energy'] = energy_tensor.detach().cpu().item()
            self.results['forces'] = forces_tensor.detach().cpu().numpy()

Since the library hasn't been written with the same optimization of the native library, using c++ and parallelization to obtain a very fast algorithm, we try to add a repulsive potential in order to easy the job of the MACE computation and compensating the role that the unoptimized simpler "MACE" cannot satisfy with  this few steps and windows

In [ ]:
import ase.io  # <-- The fix
import ase.units as units
import numpy as np
import os
from google.colab import drive
from ase.calculators.mixing import SumCalculator
from ase.calculators.lj import LennardJones
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.io.trajectory import Trajectory

# ==========================================
# PHASE 4: Run Stabilized Molecular Dynamics
# ==========================================
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define all file paths pointing to the MDProject folder
base_dir = '/content/drive/MyDrive/MDProject'

outcar_path = os.path.join(base_dir, 'OUTCAR')
cleaned_path = os.path.join(base_dir, 'cleaned_dataset.extxyz')
train_path = os.path.join(base_dir, 'train.extxyz')
valid_path = os.path.join(base_dir, 'valid.extxyz')

print("\n--- Starting Molecular Dynamics Simulation ---")

# 1. Load starting structure
md_atoms = ase.io.read(valid_path, index=0)

# 2. Attach the PyTorch Brain + Classical Repulsive Core
mace_calc = CustomMACECalculator(model, device)

# Create the Repulsive Wall (sigma=1.2 A threshold, epsilon=0.001 eV depth)
repulsive_calc = LennardJones(sigma=1.0, epsilon=0.001)

# SumCalculator automatically adds the energies and forces of both
md_atoms.calc = SumCalculator([mace_calc, repulsive_calc])

# --- SANITY CHECK ---
initial_energy = md_atoms.get_potential_energy()
max_force = np.max(np.abs(md_atoms.get_forces()))

print(f"Sanity Check - Initial Combined Energy: {initial_energy:.3f} eV")
print(f"Sanity Check - Max Force on any atom: {max_force:.3f} eV/Å")

# 3. Initialize thermal velocities for 300 K
temperature_K = 300.0
MaxwellBoltzmannDistribution(md_atoms, temperature_K=temperature_K)

# 4. Stabilized MD Engine Setup (0.5 fs steps)
timestep = 0.5 * units.fs
friction = 0.01

dyn = Langevin(md_atoms, timestep, temperature_K * units.kB, friction)

# 5. Setup Trajectory Saving
os.makedirs(f"{base_dir}/results", exist_ok=True)
traj_path = os.path.join(base_dir, 'results', 'MACEscratch2_simulation.traj')
traj = Trajectory(traj_path, 'w', md_atoms)
dyn.attach(traj.write, interval=1)

def print_dynamics():
    epot = md_atoms.get_potential_energy()
    ekin = md_atoms.get_kinetic_energy()
    temp = ekin / (1.5 * units.kB * len(md_atoms))
    current_max_f = np.max(np.abs(md_atoms.get_forces()))

    print(f"Step: {dyn.get_number_of_steps():>4} | Temp: {temp:>6.1f} K | Max Force: {current_max_f:>5.1f} eV/Å | Total E: {epot+ekin:.2f} eV")

dyn.attach(print_dynamics, interval=20)

# 6. RUN THE SIMULATION
print("\nRunning 500 steps of simulation time...")
dyn.run(500)

print(f"\nSimulation complete! Trajectory saved to: {traj_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Starting Molecular Dynamics Simulation ---


NameError: name 'model' is not defined

In [ ]:
import ase.io
import ase.units as units
import numpy as np
import os
import torch # <-- Added torch to load the model
from google.colab import drive
from ase.calculators.mixing import SumCalculator
from ase.calculators.lj import LennardJones
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.io.trajectory import Trajectory

# ==========================================
# PHASE 4: Run Stabilized Molecular Dynamics
# ==========================================
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define all file paths pointing to the MDProject folder
base_dir = '/content/drive/MyDrive/MDProject'

outcar_path = os.path.join(base_dir, 'OUTCAR')
cleaned_path = os.path.join(base_dir, 'cleaned_dataset.extxyz')
train_path = os.path.join(base_dir, 'train.extxyz')
valid_path = os.path.join(base_dir, 'valid.extxyz')
model_path = os.path.join(base_dir, 'checkpoints', 'custom_mace_final.pt') # <-- Path to saved weights

print("\n--- Starting Molecular Dynamics Simulation ---")

# 3. Setup device and load the saved PyTorch Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the empty MACE architecture (must match your training parameters)
model = MACEForceWrapper(
    num_elements=100,
    cutoff=5.0,
    lmax=0,
    correlation_order=3,
    num_layers=2
).to(device)

# Load the trained weights into the model
print(f"Loading trained weights from {model_path}...")
model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model.eval() # Set to evaluation mode for inference

# 4. Load starting structure
md_atoms = ase.io.read(valid_path, index=0)

# 5. Attach the PyTorch Brain + Classical Repulsive Core
mace_calc = CustomMACECalculator(model, device)

# Create the Repulsive Wall (sigma=1.0 A threshold, epsilon=0.001 eV depth)
repulsive_calc = LennardJones(sigma=1.0, epsilon=0.001)

# SumCalculator automatically adds the energies and forces of both!
md_atoms.calc = SumCalculator([mace_calc, repulsive_calc])

# --- SANITY CHECK ---
initial_energy = md_atoms.get_potential_energy()
max_force = np.max(np.abs(md_atoms.get_forces()))

print(f"Sanity Check - Initial Combined Energy: {initial_energy:.3f} eV")
print(f"Sanity Check - Max Force on any atom: {max_force:.3f} eV/Å")

# 6. Initialize thermal velocities for 300 K
temperature_K = 300.0
MaxwellBoltzmannDistribution(md_atoms, temperature_K=temperature_K)

# 7. Stabilized MD Engine Setup (0.5 fs steps)
timestep = 0.5 * units.fs
friction = 0.01

dyn = Langevin(md_atoms, timestep, temperature_K * units.kB, friction)

# 8. Setup Trajectory Saving
os.makedirs(f"{base_dir}/results", exist_ok=True)
traj_path = os.path.join(base_dir, 'results', 'New_simulation.traj')
traj = Trajectory(traj_path, 'w', md_atoms)
dyn.attach(traj.write, interval=1)

def print_dynamics():
    epot = md_atoms.get_potential_energy()
    ekin = md_atoms.get_kinetic_energy()
    temp = ekin / (1.5 * units.kB * len(md_atoms))
    current_max_f = np.max(np.abs(md_atoms.get_forces()))

    print(f"Step: {dyn.get_number_of_steps():>4} | Temp: {temp:>6.1f} K | Max Force: {current_max_f:>5.1f} eV/Å | Total E: {epot+ekin:.2f} eV")

dyn.attach(print_dynamics, interval=2)

# 9. RUN THE SIMULATION
print("\nRunning 500 steps of simulation time...")
dyn.run(500)

print(f"\nSimulation complete! Trajectory saved to: {traj_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Starting Molecular Dynamics Simulation ---
Loading trained weights from /content/drive/MyDrive/MDProject/checkpoints/custom_mace_final.pt...
Sanity Check - Initial Combined Energy: -0.219 eV
Sanity Check - Max Force on any atom: 0.600 eV/Å

Running 500 steps of simulation time...
Step:    0 | Temp:  299.2 K | Max Force:   0.6 eV/Å | Total E: 3.65 eV


/tmp/ipykernel_1252/1521911014.py:68: DeprecationWarning: Use thermalize_momenta
  MaxwellBoltzmannDistribution(md_atoms, temperature_K=temperature_K)
/usr/local/lib/python3.13/dist-packages/ase/md/md.py:54: FutureWarning: Specify the temperature in K using the 'temperature_K' argument
  warnings.warn(FutureWarning(w))
/usr/local/lib/python3.13/dist-packages/ase/md/langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


Step:    2 | Temp:  289.3 K | Max Force:   0.9 eV/Å | Total E: 3.66 eV
Step:    4 | Temp:  265.3 K | Max Force:   1.2 eV/Å | Total E: 3.67 eV
Step:    6 | Temp:  243.0 K | Max Force:   1.3 eV/Å | Total E: 3.66 eV
Step:    8 | Temp:  235.5 K | Max Force:   1.2 eV/Å | Total E: 3.67 eV
Step:   10 | Temp:  237.7 K | Max Force:   1.2 eV/Å | Total E: 3.66 eV
Step:   12 | Temp:  239.2 K | Max Force:   1.2 eV/Å | Total E: 3.65 eV
Step:   14 | Temp:  231.5 K | Max Force:   1.3 eV/Å | Total E: 3.65 eV
Step:   16 | Temp:  216.4 K | Max Force:   1.6 eV/Å | Total E: 3.65 eV
Step:   18 | Temp:  202.9 K | Max Force:   1.6 eV/Å | Total E: 3.66 eV
Step:   20 | Temp:  197.3 K | Max Force:   1.6 eV/Å | Total E: 3.65 eV
Step:   22 | Temp:  197.2 K | Max Force:   1.7 eV/Å | Total E: 3.65 eV
Step:   24 | Temp:  194.5 K | Max Force:   1.6 eV/Å | Total E: 3.67 eV
Step:   26 | Temp:  185.0 K | Max Force:   1.8 eV/Å | Total E: 3.70 eV
Step:   28 | Temp:  174.7 K | Max Force:   1.8 eV/Å | Total E: 3.70 eV
Step: 

## Library loading

In [ ]:
! pip install e3nn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 15.7 MB/s eta 0:00:00


In [ ]:
! pip install ase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 27.4 MB/s eta 0:00:00


In [ ]:
! pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.1 MB/s eta 0:00:00


In [ ]:
! pip install torch_ema

In [ ]:
import torch
import numpy as np
from ase.io import read
from torch_geometric.data import Data, DataLoader
from torch.optim import AdamW
from torch_ema import ExponentialMovingAverage

def load_extxyz_to_pyg(file_path: str):
    """Parses an .extxyz file and converts it into a list of PyTorch Geometric graphs."""
    # Read all atomic structures from the file
    structures = read(file_path, index=":")
    dataset = []

    for atoms in structures:
        # Extract features
        positions = torch.tensor(atoms.get_positions(), dtype=torch.float32)
        atomic_numbers = torch.tensor(atoms.get_atomic_numbers(), dtype=torch.long)

        # Extract periodic boundary conditions and simulation cell
        cell = torch.tensor(atoms.get_cell().array, dtype=torch.float32)
        pbc = torch.tensor(atoms.get_pbc(), dtype=torch.bool)

        # Extract ground-truth targets safely handling ASE warnings
        energy_val = atoms.info.get('REF_energy', atoms.info.get('energy', 0.0))
        energy = torch.tensor([energy_val], dtype=torch.float32)

        if 'REF_forces' in atoms.arrays:
            forces_val = atoms.arrays['REF_forces']
        elif 'forces' in atoms.arrays:
            forces_val = atoms.arrays['forces']
        else:
            forces_val = atoms.get_forces()
        forces = torch.tensor(forces_val, dtype=torch.float32)

        # Package into a PyG Data object
        data = Data(
            positions=positions,
            atomic_numbers=atomic_numbers,
            cell=cell,
            pbc=pbc,
            REF_energy=energy,
            REF_forces=forces
        )
        dataset.append(data)

    return dataset

In [ ]:
from torch.optim import AdamW
from torch_ema import ExponentialMovingAverage
from torch.optim.swa_utils import AveragedModel, SWALR
optimizer = AdamW(
    model.parameters(),
    lr=0.0001,             # --lr=0.0001
    weight_decay=5e-7,     # --weight_decay=5e-7
    amsgrad=True           # --amsgrad
)

# The logs indicate gradient clipping is applied with a tolerance of 10.0
max_grad_norm = 10.0

#  Initialize EMA with a decay of 0.99
ema = ExponentialMovingAverage(model.parameters(), decay=0.99)

#  Initialize SWA
swa_model = AveragedModel(model)
swa_start_epoch = 30
swa_scheduler = SWALR(optimizer, swa_lr=0.001) # swa_lr=0.001

NameError: name 'model' is not defined